In [ ]:
from symbolica import Expression, S, E
from symbolica.community.spenso import (
    AUTO, BroadcastFunction, Representation, Slot, Tensor,
    TensorExpression, TensorFunctionLibrary, TensorLibrary, TensorName as N,
    TensorNetwork, chain, dot, trace,
)

from IPython.display import display
import random

# Tensor structures and representations:

A tensor starts with a structure declaring its ordered representation ports. Addressed ports are `Slot` values; unresolved ports remain `Representation` values in the `TensorExpression.interface`.

Representations are defined by their name and dimension. They can be self-dual or not.


In [ ]:
mink = Representation.mink(4)
bis = Representation.bis(4)
custom = Representation("custom",4,is_self_dual=False)
print(custom)

Slots are created from a representation and an index

In [ ]:
mu = mink("mu")
print(mu)
mu

They can be converted to symbolica expressions

In [ ]:
mue = mu.to_expression()
mue


In [ ]:
nu = Slot("mink",4,"nu")

In [ ]:
# The index can be a string (that could be parsed into a symbolica symbol)
i = bis("i")
# The index can also be an integer
j = bis(2)

k = S("k")
# The index can also directly a symbolica expression
k = bis(k)
print(k)
k

In [ ]:
gamma = N.gamma()
p = N.vector("P")
w = N.vector("w")
g = N.g()
mq = S("mq")

Calling a `TensorName` always creates a `TensorExpression`. Its interface contains explicit `Slot` values and unresolved `Representation` values in logical order. Calling or indexing the expression later addresses only ports left unresolved with `_` or its cross-cell-friendly alias `AUTO`.


In [ ]:

gamma_structure = gamma(mink, bis, bis)
open_gamma = gamma_structure(mu, AUTO, AUTO)
g_muik = open_gamma(i, k)
print(g_muik)
print(g_muik.interface)

In [ ]:
g_muik[2]

In [ ]:
print(gamma_structure(nu, k, j).to_expression())

In [ ]:

g_muik[45:63:3]


In [ ]:
g_muik[[2,2,2]]

Tensor-aware multiplication chooses dot products and matrix composition when the channel is unique. The explicit helpers remain available when you want to state the topology directly.


In [ ]:
square = p(1, mink) * p(1, mink)
factors = (gamma_structure(mu, AUTO, AUTO), gamma_structure(nu, AUTO, AUTO))
open_chain = chain(bis("a"), bis("b"), *factors)
dirac_trace = trace(bis, *factors)
print(square.format_tensor())
print(open_chain.format_tensor())
print(dirac_trace.format_tensor())
display(dirac_trace.formatted())
print(dirac_trace.to_typst())

In [ ]:
x = (
    g_muik
    * (p(2, nu) * gamma_structure(nu, k, j) + mq * g(k, j))
    * w(1, i)
    * w(3, mu)
)

print(x.to_canonical_string())

In [ ]:
tn = x.to_network()
# prints the rich graph associated to the network
print(tn)

As you can see when parsed, the network isn't contracted yet, so it's just a graph of the expression.
Call `execute` to contract it; optional arguments select how much of the network is evaluated.


In [ ]:
from symbolica.community.spenso import ExecutionMode
tn.execute(n_steps=2,mode=ExecutionMode.Scalar)


In [ ]:
t = TensorNetwork.one()*TensorNetwork.zero() + TensorNetwork.one()*TensorNetwork.zero()
print(t)
t.execute()
print(t)

The graph has now reduced in size

In [ ]:
print(tn)


We can now extract the resulting tensor, by fully executing the network


In [ ]:
tn.execute()
t = tn.result_tensor()


The tensor is a tensor object


In [ ]:
t


 It has a structure


In [ ]:
print(t.structure())


# Evaluation of the tensor

 You may have noticed that the resulting tensor is a set of expressions with certain functions that label the 'concrete' values of the tensor. What if we want to evaluate the tensor for a given set of parameters?


In [ ]:

params = [
    Expression.I,
    *w(1, i).to_network().result_tensor(),  # tensors implement the sequence protocol
    *w(3, mu).to_network().result_tensor(),
    *p(2, nu).to_network().result_tensor(),
]
constants = {mq: E("173")}

# Much like the expressions, tensors have the same evaluation api, just that they return a tensor instead of an expression
e=t.evaluator(constants=constants, params=params, funs={})
# The evaluator can be compiled to a shared library
c = e.compile(function_name="f", filename="test_expression.cpp",
              library_name="test_expression.so", inline_asm="none")


e_params = [random.random()+1j*random.random() for i in range(len(params))]
eval_res = e.evaluate_complex([e_params])[0]

print(eval_res)
print(eval_res.structure())



# Tensor building:
Tensors have dense and hashmap-backed sparse storage. Both constructors require a named `TensorExpression`; a name can be attached to an atomic or composite expression without changing its symbolic value.


In [ ]:
small_euc = Representation.euc(2)
small_mink = Representation.mink(3)
row = N.vector("stored_row")
column = N.vector("stored_column")
dense_descriptor = row(small_euc("row")).outer(
    column(small_mink("column"))
).with_name(N("stored_matrix"))
dense_tensor = Tensor.dense(dense_descriptor, list(range(6)))
sparse_descriptor = N.vector("stored_sparse")(small_euc("entry"))
sparse_tensor = Tensor.sparse(sparse_descriptor, float)
sparse_tensor[1] = 2.0
print(dense_tensor[[1, 2]], sparse_tensor[:])
display(dense_tensor.formatted())
print(dense_tensor.to_typst())
# Concrete arithmetic promotes to a network with the same inference rules.
data_network = dense_tensor * sparse_tensor
print(data_network)

In [ ]:


sparse_descriptor = N("custom_sparse")(custom, custom)
t = Tensor.sparse(sparse_descriptor, type(mq))
# Stored data always exposes a named TensorExpression descriptor.
print(t.structure())


The unresolved representations define the logical data layout used for storage and iteration. The tensor can store expressions, floats, or complex numbers homogeneously. Values are accessed by flattened logical index (and slices are supported).


In [ ]:
t[6]=E("f(x)*(1+y)")
t

Or by multi-index:


In [ ]:
t[[3,2]]=E("sin(alpha)")
t


Dense and sparse tensors share the same semantic display API. Converting to dense changes storage, not presentation.

In [ ]:
t.to_dense()
t


# Registering to a library

In [ ]:

d = Representation("newrep",3)
left = N.vector("library_left")
right = N.vector("library_right")
definition = left(d).outer(right(d)).with_name(
    N("test")
)
# Dense tensors are built from a named TensorExpression in logical row-major order.
t = Tensor.dense(definition, [0, 0, 123,
                                     11, 3, 234,
                                     234, 23, 44])
t[[1,2]]=3/34
print(t)
print(t.structure())

lib = TensorLibrary.hep_lib()

lib.register(t)

composite_definition = t.structure()
new_t = lib[definition]  # precise lookup by name, scalar args, and representations
print(composite_definition.to_expression())
print(new_t.to_expression())


x = new_t(1, 2) * new_t(2, 3) * new_t(3, 1)
n = x.to_network(library=lib)
n.execute(library=lib)
t = n.result_tensor(library=lib)
print(t)


# Elementwise broadcast functions

Broadcasting is symbolic by default. Concrete network execution uses a callback registered in a `TensorFunctionLibrary`.


In [ ]:
sample = N.vector("broadcast_sample")
sample_descriptor = sample(d)
sample_tensor = Tensor.dense(sample_descriptor, [1.0, 4.0, 9.0])
sqrt = BroadcastFunction("elementwise_sqrt", is_real=True)
functions = TensorFunctionLibrary()
functions.register(sqrt, lambda value: value**0.5)
sqrt_network = sqrt(sample_tensor)
sqrt_network.execute(function_library=functions)
print(sqrt_network.result_tensor())

# Algebraic simplification

In [ ]:
minkd = Representation.mink("D")
coad = Representation.coad(8)
cof = Representation.cof(3)

gamma_d = N.gamma()(minkd, bis, bis)
momentum = N.vector("simplification_p")
color_f = N.f()
print(coad.dimension, coad.casimir())


In [ ]:
from symbolica.community import idenso

metric_trace = minkd.g(1, 1)
print(metric_trace.simplify_metrics())
# Module functions deliberately return an ordinary Expression.
print(idenso.simplify_metrics(metric_trace.to_expression()))

In [ ]:
bis.g(1, 1).simplify_metrics()

In [ ]:
Representation.euc("d").g(1, 1).simplify_metrics()

In [ ]:
a = (
    gamma_d("mu", 1, 2)
    * gamma_d("nu", 2, 3)
    * gamma_d("rho", 3, 4)
    * gamma_d("sigma", 4, 1)
    * momentum(minkd("mu"))
    * momentum(minkd("nu"))
    * momentum(minkd("rho"))
    * momentum(minkd("sigma"))
).simplify_gamma()

In [ ]:
print(a.to_dots())

In [ ]:
color_scalar = color_f(coad(1), coad(2), coad(3)).contract(
    color_f(coad(3), coad(2), coad(1)), left=0, right=2
)
# Undoing the scalar dot is intentionally requested as a base-expression escape hatch.
idenso.simplify_color(idenso.undo_dots(color_scalar.to_expression()))